## Install `holidays`

The Databricks Serverless environment does not ship the `holidays` package. This cell installs it and restarts Python so `src.features` can import it. Only takes ~5 seconds.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 06 — Live API ingestion (AviationStack)

Fetches a live flight, validates against the Silver contract, and writes
`api_bronze_flights` and `api_silver_flights`. Falls back to a committed
fixture when `USE_FIXTURE=True`, so the pipeline stays demoable when the
key is missing or the 100-request free quota is spent.

In [0]:
import sys
sys.path.append("..")

import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

from src import config
from src.api_pipeline import (
    LookupParams, fetch_flights, load_fixture, project_to_silver, validate_schema,
)

## Runtime switches

In [0]:
dbutils.widgets.dropdown("USE_FIXTURE", "false", ["true", "false"])
dbutils.widgets.text("DEP_IATA", "ATL")
dbutils.widgets.text("ARR_IATA", "LAX")
dbutils.widgets.text("AIRLINE_IATA", "")

USE_FIXTURE = dbutils.widgets.get("USE_FIXTURE").lower() == "true"
params = LookupParams(
    dep_iata=dbutils.widgets.get("DEP_IATA") or None,
    arr_iata=dbutils.widgets.get("ARR_IATA") or None,
    airline_iata=dbutils.widgets.get("AIRLINE_IATA") or None,
)

## Fetch

In [0]:
if USE_FIXTURE:
    payload = load_fixture()
    print(f"Using fixture ({len(payload.get('data', []))} records)")
else:
    access_key = dbutils.secrets.get(
        scope=config.AVIATIONSTACK_SECRET_SCOPE,
        key=config.AVIATIONSTACK_SECRET_KEY,
    )
    payload = fetch_flights(params, access_key)
    print(f"AviationStack returned {len(payload.get('data', []))} records")

Using fixture (2 records)


## Write API Bronze (raw JSON payload as string)

In [0]:
import json as _json

raw_row = [(_json.dumps(payload),)]
bronze_df = (
    spark.createDataFrame(raw_row, ["raw_payload"])
    .withColumn("ingested_at", current_timestamp())
    .withColumn("used_fixture", lit(USE_FIXTURE))
)
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.API_BRONZE)
)
print(f"Appended to {config.API_BRONZE}")

Appended to workspace.flights.api_bronze_flights


## Project to Silver + DQ check

In [ ]:
silver_pdf = project_to_silver(payload)
report = validate_schema(silver_pdf)
print(report)

if not report["passed"]:
    raise ValueError(f"Data-quality gate failed: {report}")

# Deliberately NOT written here. flight_phase is attached further down from the
# OpenSky snapshot, and writing first meant the column never reached the table —
# 07_score then reported it missing on every run. The write happens once, after
# the frame is complete.
print(f"Projected {len(silver_pdf)} rows. Write deferred until phase is attached.")


---

## Live aircraft state (OpenSky) — one call for the whole airspace

AviationStack is queried per origin/destination pair. With 7,675 distinct routes in the
historical data, covering them once costs 7,675 calls, and refreshing costs that again every
cycle. That does not improve with a paid tier; the shape is wrong.

OpenSky's `/states/all` returns every aircraft the network is tracking in a single request.
Measured against the continental US box: **8,166 aircraft in one 1.1 MB response**, 2,893 of
them carrying callsigns belonging to carriers present in the BTS data. Call volume stops
scaling with the number of flights and scales only with polling frequency.

**The division of labour.** OpenSky decides *which* flights are worth asking about;
AviationStack answers *how late* they were.

That split is not arbitrary. The model trains on BTS `DEP_DELAY`, which is **gate** departure
delay. OpenSky observes when an airframe stops being `on_ground`, which is **wheels-off**.
They differ by taxi-out — minutes at a small field, far more at a congested hub — so deriving
delay from OpenSky would inject a bias that is worst precisely where delay matters most, and
would do it silently. AviationStack's `departure.delay` is already gate semantics, so that is
the value served. `docs/API_STRATEGY.md` has the full argument.

**The join needs no mapping table.** AviationStack returns `flight.icao` (`"DAL1234"`);
OpenSky broadcasts `callsign` in the same ICAO form, space-padded. After stripping they
compare directly, because AviationStack supplies the ICAO spelling itself.

**What `on_ground` buys.** It is the phase split the two models need, arriving free with data
already fetched. `True` is the pre-departure population, `False` is in-flight. Without it both
models score every flight regardless of whether it has left the gate, which is incoherent.


In [0]:
from src.opensky import (
    CONUS_BBOX, OpenSkyClient, match_to_schedule, parse_states, split_by_phase,
)

dbutils.widgets.dropdown("USE_OPENSKY", "true", ["true", "false"])
USE_OPENSKY = dbutils.widgets.get("USE_OPENSKY") == "true"

state_rows, phase_by_icao, opensky_report = [], {}, None

if not USE_OPENSKY:
    print("USE_OPENSKY=false — skipping the live state feed.")
else:
    try:
        # The client id and secret are long-lived and live in Databricks secrets.
        # The bearer token minted from them lasts 1,800 seconds, so the client
        # refreshes on demand rather than the notebook handling tokens at all.
        # Nothing here ever needs re-saving to the secret scope.
        opensky = OpenSkyClient(
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_ID_KEY),
            dbutils.secrets.get(config.OPENSKY_SECRET_SCOPE, config.OPENSKY_CLIENT_SECRET_KEY),
        )

        payload_os = opensky.fetch_states(CONUS_BBOX)
        state_rows = parse_states(payload_os)
        ground, airborne = split_by_phase(state_rows)

        print(f"OpenSky: {len(state_rows):,} aircraft in ONE call "
              f"(token refreshes: {opensky.refresh_count})")
        print(f"  on ground : {len(ground):,}   (pre-departure population)")
        print(f"  airborne  : {len(airborne):,}   (in-flight population)")

        scheduled_icao = [c for c in silver_pdf.get("flight_icao", []) if c]
        matched, opensky_report = match_to_schedule(state_rows, scheduled_icao)

        print(f"\nMatched against {opensky_report['scheduled_flights']} scheduled flights:")
        for k, v in opensky_report.items():
            shown = f"{v:.1%}" if isinstance(v, float) else v
            print(f"  {k:<26} {shown}")

        phase_by_icao = {
            r["callsign"]: ("airborne" if r["on_ground"] is not True else "on_ground")
            for r in matched
        }

        print("\nA low match rate is expected and is not by itself a defect. Around 39% of")
        print("aircraft aloft over the US are N-registered general aviation that will never")
        print("appear in a BTS schedule, and a scheduled flight only appears in the snapshot")
        print("if it happens to be moving right now — most of a day's schedule is not.")
        print("Measure the rate, publish it, do not assume it.")
    except Exception as e:
        # The live feed is an enhancement, not a dependency. Bronze -> Gold ->
        # train never touches an API, and scoring falls back to pre-departure.
        print(f"OpenSky unavailable ({type(e).__name__}: {e}).")
        print("Continuing without the phase split — every flight falls back to pre-departure.")


OpenSky: 2,684 aircraft in ONE call (token refreshes: 1)
  on ground : 319   (pre-departure population)
  airborne  : 2,365   (in-flight population)

Matched against 2 scheduled flights:
  states_seen                2684
  with_callsign              2657
  without_callsign           27
  scheduled_flights          2
  matched                    0
  match_rate_of_scheduled    0.0%

A low match rate is expected and is not by itself a defect. Around 39% of
aircraft aloft over the US are N-registered general aviation that will never
appear in a BTS schedule, and a scheduled flight only appears in the snapshot
if it happens to be moving right now — most of a day's schedule is not.
Measure the rate, publish it, do not assume it.


### Landing the snapshot and attaching phase

The raw snapshot goes to its own Delta table so a scoring run can be reconstructed later —
what the airspace looked like at the moment a prediction was made is exactly the evidence you
need to audit that prediction afterwards.

`flight_phase` carries three values, and the third is the honest one:

- `airborne` — OpenSky sees it flying, so it has departed and `dep_delay` is real
- `on_ground` — OpenSky sees it, still at the field
- `unknown` — not matched. No callsign, outside the box, or general aviation. These fall back
  to the pre-departure model, which is the safe default: it is the variant that does not
  require a departure to have happened.


In [ ]:
if state_rows:
    states_sdf = (
        spark.createDataFrame(pd.DataFrame(state_rows))
        .withColumn("ingested_at", current_timestamp())
    )
    (
        states_sdf.write.format("delta").mode("append")
        .option("mergeSchema", "true").saveAsTable(config.OPENSKY_STATES)
    )
    print(f"Appended {len(state_rows):,} state vectors -> {config.OPENSKY_STATES}")

# Attach phase, THEN write. The previous ordering wrote api_silver two cells
# earlier and set flight_phase on a frame nothing read afterwards.
silver_pdf["flight_phase"] = (
    silver_pdf["flight_icao"].map(phase_by_icao).fillna("unknown")
    if "flight_icao" in silver_pdf.columns and phase_by_icao
    else "unknown"
)
counts = silver_pdf["flight_phase"].value_counts().to_dict()
print(f"\nPhase assignment across {len(silver_pdf)} scheduled flights: {counts}")

silver_sdf = (
    spark.createDataFrame(silver_pdf)
    .withColumn("ingested_at", current_timestamp())
)
(
    silver_sdf.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.API_SILVER)
)
print(f"Appended {silver_sdf.count()} rows to {config.API_SILVER} (with flight_phase)")

# dep_delay only means anything once the aircraft has actually left. AviationStack
# reports it on gate semantics, matching what the model trained on.
has_delay = silver_pdf["dep_delay"].notna().sum()
print(f"\nRows carrying an AviationStack gate dep_delay: {has_delay} of {len(silver_pdf)}")
print("Those are the rows the in-flight model can score. The rest get the")
print("pre-departure model, which is the only one that works before pushback.")


## Log DQ result to the data-quality table

In [0]:
from pyspark.sql import Row

dq_row = spark.createDataFrame([
    Row(
        checked_at=report["checked_at"],
        source="aviationstack",
        used_fixture=USE_FIXTURE,
        row_count=report["row_count"],
        passed=report["passed"],
        missing_columns=",".join(report["missing_columns"]),
    )
])
(
    dq_row.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.DATA_QUALITY_LOG)
)